In [45]:
import sqlite3
import pandas as pd

df=pd.read_csv("world_happiness_dataset.csv")

conn = sqlite3.connect("happiness.db")

df.to_sql("happiness", conn, if_exists="replace", index=False)

print("Database created and data inserted!")

Database created and data inserted!


In [46]:
pd.read_sql("""
PRAGMA table_info(happiness);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Country,TEXT,0,None,0
1,1,Happiness_Score,REAL,0,None,0
2,2,GDP_per_Capita,REAL,0,None,0
3,3,Social_Support,REAL,0,None,0
4,4,Healthy_Life_Expectancy,REAL,0,None,0
5,5,Freedom_to_Make_Choices,REAL,0,None,0
6,6,Generosity,REAL,0,None,0
7,7,Perceptions_of_Corruption,REAL,0,None,0


In [47]:
pd.read_sql("""
SELECT * 
FROM happiness limit 10;
""", conn)

,Country,Happiness_Score,GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom_to_Make_Choices,Generosity,Perceptions_of_Corruption
0,Norway,6.25,1.39,0.82,81.6,0.69,0.01,0.71
1,Denmark,3.61,1.27,0.43,66.9,0.48,0.43,0.53
2,Iceland,4.68,0.87,0.54,63.4,0.71,0.41,0.72
3,Switzerland,4.46,0.67,0.57,68.8,0.93,0.32,0.52
4,Finland,6.67,1.55,0.45,75.4,0.58,0.16,0.10
5,Netherlands,6.41,0.87,0.54,72.6,0.45,0.38,0.36
6,Canada,7.34,0.60,0.46,49.6,1.00,0.07,0.12
7,New Zealand,3.87,0.61,0.57,41.3,0.66,0.26,0.84
8,Australia,5.31,1.43,0.78,53.2,0.36,0.27,0.80
9,Sweden,3.63,1.16,0.62,51.2,0.33,0.57,0.77


In [48]:
query1 = """
WITH categorized AS (
    SELECT
        Country,
        Happiness_Score,
        GDP_per_Capita,

        CASE
            WHEN GDP_per_Capita < 0.9 THEN 'Low'
            WHEN GDP_per_Capita BETWEEN 0.9 AND 1.3 THEN 'Medium'
            ELSE 'High'
        END AS GDP_Category

    FROM happiness
),

avg_table AS (
    SELECT
        GDP_Category,
        AVG(Happiness_Score) AS avg_happiness
    FROM categorized
    GROUP BY GDP_Category
),

ranked AS (
    SELECT
        Country,
        GDP_Category,
        Happiness_Score,

        RANK() OVER (
            PARTITION BY GDP_Category
            ORDER BY Happiness_Score DESC
        ) AS rank_in_group

    FROM categorized
)

SELECT
    r.Country,
    r.GDP_Category,
    r.Happiness_Score,
    r.rank_in_group,
    a.avg_happiness
FROM ranked r
JOIN avg_table a
ON r.GDP_Category = a.GDP_Category
ORDER BY r.GDP_Category, r.rank_in_group;
"""

pd.read_sql(query1, conn)

,Country,GDP_Category,Happiness_Score,rank_in_group,avg_happiness
0,Brazil,High,6.98,1,5.494286
1,Finland,High,6.67,2,5.494286
2,Norway,High,6.25,3,5.494286
3,Australia,High,5.31,4,5.494286
4,India,High,4.45,5,5.494286
5,United States,High,4.44,6,5.494286
6,France,High,4.36,7,5.494286
7,Canada,Low,7.34,1,5.352000
8,Netherlands,Low,6.41,2,5.352000
9,Iceland,Low,4.68,3,5.352000


In [49]:
query2 = """
WITH corruption_group AS (
    SELECT
        Country,
        Happiness_Score,
        GDP_per_Capita,
        Healthy_Life_Expectancy,
        Freedom_to_Make_Choices,
        Perceptions_of_Corruption,

        CASE
            WHEN Perceptions_of_Corruption >= 0.5 THEN 'High Corruption'
            ELSE 'Low Corruption'
        END AS corruption_level

    FROM happiness
)

SELECT
    corruption_level,

    AVG(Happiness_Score) AS avg_happiness,
    AVG(GDP_per_Capita) AS avg_gdp,
    AVG(Healthy_Life_Expectancy) AS avg_health,
    AVG(Freedom_to_Make_Choices) AS avg_freedom

FROM corruption_group
GROUP BY corruption_level;
"""

pd.read_sql(query2, conn)

,corruption_level,avg_happiness,avg_gdp,avg_health,avg_freedom
0,High Corruption,4.799091,1.153636,60.618182,0.660000
1,Low Corruption,5.626667,1.143333,64.333333,0.662222
